# 📚 MODULE 24 — MODERN CNN DESIGN PRINCIPLES

# 🧭 1. The Big Picture

Modern CNN design can be understood through **three layers of thinking**:

```text
                    CNN DESIGN
                        │
        ┌───────────────┼────────────────┐
        ↓               ↓                ↓
  BUILDING BLOCKS   DESIGN DIMENSIONS   TRADE-OFFS
        │               │                │
        ↓               ↓                ↓
  Bottleneck          Depth           Accuracy
  Skip                Width           Parameters
  Dense               Resolution      FLOPs
  Group Conv          Receptive       Memory
  Depthwise Conv      Field            Latency
  1×1 Conv
  Large Kernels
  GAP
```

This is the framework you should use when analyzing **any modern CNN**.

---

# 2. Bottleneck 🔻

## The problem

Suppose we have:

$$
C=256
$$

channels and want to perform an expensive \(3\times3\) convolution.

A standard \(3\times3\) convolution from 256 → 256 requires:

$$
3\times3\times256\times256
$$

$$
=\boxed{589,824}
$$

parameters.

Instead, we can first compress the representation:

```text
256 channels
      ↓
   1×1 Conv
      ↓
64 channels
      ↓
   3×3 Conv
      ↓
64 channels
      ↓
   1×1 Conv
      ↓
256 channels
```

This is the **bottleneck principle**.

---

## Why does it work?

The expensive spatial operation now operates on 64 channels:

$$
3\times3\times64\times64
=
36,864
$$

instead of:

$$
589,824
$$

So the central idea is:

$$
\boxed{
\text{compress}
\rightarrow
\text{expensive transformation}
\rightarrow
\text{expand}
}
$$

The \(1\times1\) convolutions make this practical.

---

## 🧠 Mental model

Think of it like this:

```text
Wide representation
       ↓
  Squeeze information
       ↓
Process cheaply
       ↓
Restore dimensions
```

Bottleneck blocks became particularly important in ResNet-style architectures and subsequently many CNN families.

---

# 3. Skip / Residual Connection 🔄

The standard transformation is:

$$
Y=F(X)
$$

A residual block instead uses:

$$
\boxed{Y=F(X)+X}
$$

Visual:

![Image](https://images.openai.com/static-rsc-4/YdkPyghfveCpUwz7HhfNKn4c5TAcnPMPiI87I9fcbWFquWjaDyX6wqu0CnDww6eAX1uX6cMJ5RYlUuTDOBBf6XOlPvhsQboeoQ-jXfbazW1WPwrABSe8j3X7VunoRj95yeq_ZHqvbCyOjzzXs9y7WA4g_UEsokiOsP8WZWxJtv-4tbReAqMQrEf59XyP0LrK?purpose=fullsize)

![Image](https://images.openai.com/static-rsc-4/0Y6B-L9Z_PeMY3lpjwgqGsv3FkMcVIxvdJ6Qm2OewtznM83oAs-8e8rebmbSGCnDrLVbhUOG7qfTXWIE9F0wpFJbYZREQ9zZlA-NeP5VuxMmbGty712jlCrW5YieMw_Pg5cnU9wHS4KoIGWsSO5c0KZ3CpsqZ3UBkidy9d_uSTqAylkB6hufpGr55XcxWLae?purpose=fullsize)

![Image](https://images.openai.com/static-rsc-4/w05-EKzt7NIW5erF3MGSZIPj1EvWGrdVECWCYOv0wSXifgEl7t1hU3uKEDVw4TJiBwE_Gr52aUeq9UrHIRzZKlFoiUnKeBARmS9S3xUbAOe1kZKNieV7y2JqTbqUUgl-nchs086Jm_rY5nOGRcX9aj3MIoG90sS8rW10TrlaAu7WrGhHU11XDe2TQHYlu2GQ?purpose=fullsize)

![Image](https://images.openai.com/static-rsc-4/eB82Tf5HAue29O8UJrXrlY-p2Q3xJW18Icwce5MRZI6mDs1HZ1Geugz84d0pG_7huJHsj1zAZ6xCfYY_YUu5S2dhAE4Nx2VOnhY-jiLqc8HAowVJmSL8RH5nTdq2s7FdSGTAVxPb3YxW4_A1_1SJflxrNI6tvI2_LfCClOTDDqdrWQobp_EWocSSTO3i8ZnJ?purpose=fullsize)

![Image](https://images.openai.com/static-rsc-4/xofa83cigxEIUbSQya82jV81PR8TDT-Btqfkan63FE0g7HdUOPy-MUZJoBV-ghALPA6f-_G5Wf1SugkQyzrSUIeJDe0GsL6mpLfbTlz3KXd0XiyP5zJzv2WlVvmYwMSPzeFWRQaXIjXie70ffNE9V9QNZx19ih08VKANGbqevwPbbrm-tRexHRqkoFcCjmgY?purpose=fullsize)

```text
                 ┌───────────────┐
                 │               │
X ───────────────┼──→ F(X) ──────┼──→ +
                 │               ↑
                 └───────────────┘
                         X
```

The original ResNet paper introduced residual learning specifically to make substantially deeper networks easier to optimize. ([CVF Open Access][1])

---

## Why does the shortcut help?

Suppose:

$$
Y=F(X)+X
$$

Then:

$$
\frac{\partial Y}{\partial X}
=
\frac{\partial F}{\partial X}+1
$$

The derivative has a direct identity component.

That gives information and gradients a direct path through the network.

The important point is **not** that the skip connection magically prevents every optimization problem.

Rather:

> It creates a much easier path for information and gradients and lets the block learn a residual transformation.

---

# 4. Dense Connection 🕸️

Residual connection:

$$
Y=F(X)+X
$$

Dense connection goes further.

For layer \(l\):

$$
\boxed{
X_l=H_l([X_0,X_1,\ldots,X_{l-1}])
}
$$

where:

$$
[\cdot]
$$

means **concatenation**.

![Image](https://images.openai.com/static-rsc-4/BxyclWdv0vujee14X4YSl4kKI8u0FJW1S0a2b5bwXmsKswu8CW0JfPy0lIisxNQxK17lHzhs5DTN7mKtiZWULJelXn4Y_mi4-OViq5tF_9YVSLUBA6Q4q54amvNDY_vB9LYGUQBqDdmxNEyXcvTcQzOYZn-MWv7-QalotAdz87zFdOYl8yOFmvYZ4PA8rssQ?purpose=fullsize)

![Image](https://images.openai.com/static-rsc-4/fO1kiWISgEaPKVK2UWMQg6SkSB70uYXigL7AnVvm6dDCImrxy_f8uDWakG_E-DphzmfPayRg_xnkS5bZlTgAPHBveHueUkdu1XycjCgpNC7pCUQ7tGY5I14FD3hQPnrBXOWVVP5rzfRA_G-GkIhNu7VBKoKP-kGFPdX1nUS0R4bU_3dL6SBP1ZxvdrbIBPHc?purpose=fullsize)

![Image](https://images.openai.com/static-rsc-4/A6PYLwciWcKW55XOeQx1qfrMAkjiBVSHUeuPAukY8I8VbseOquCdsxo35ualypb0nrFCCT4QX8tNmWpJu_Z4I49l5MANTuVmCSSuR96h2JwwboeG0FTw3y9L3dMVwO3ADC2UfdAvD24uR3c5i0ue3OOj72EMmwC7C-hkP4Ucyt5pEiZwBr_nD7W9uTg9Dxu0?purpose=fullsize)

![Image](https://images.openai.com/static-rsc-4/8JjbKj3bBsHIhjp9VyGqMTr-JJ2GNBW1WuoWl5XGlZfLT_tqeLNawUR0-o61GmRKcSbHNRkkn_ZPCFwPdpd7ME6H8CDt_ElCzE3HWmmgXAnl-4Tm-hPmbargPB2pjy4lzmmPVFuV2X9lLYUC0uyQjvJV9RUqWNKI863d_zU_4Mca0lbCecGrJ7BYaaAhoAMB?purpose=fullsize)

Conceptually:

```text
X0 ───────────────┐
                  ↓
                 H1
                  ↓
                 X1
                  │
X0 ───────────────┼──────┐
X1 ───────────────┘      ↓
                        H2
                         ↓
                        X2
                         │
X0 ──────────────────────┼──────┐
X1 ──────────────────────┤      ↓
X2 ──────────────────────┘     H3
                                ↓
                               X3
```

DenseNet connects each layer to **every subsequent layer** in a feed-forward manner. The original paper highlights feature propagation, feature reuse, and improved gradient flow as major benefits. ([CVF Open Access][2])

---

## Residual vs Dense

### Residual

$$
F(X)+X
$$

**Combines** representations through addition.

### Dense

$$
[X_0,X_1,X_2,\ldots]
$$

**Preserves and concatenates** earlier representations.

So:

```text
ResNet
Previous representation
        +
New transformation

DenseNet
Previous representations
        ↓
Concatenate everything
```

---

# 5. Group Convolution

Suppose:

$$
C_{in}=8,\qquad C_{out}=8
$$

A normal convolution connects every input channel to every output channel:

$$
8\times8=64
$$

channel connections.

With 2 groups:

```text
Group 1:
Input 1–4 → Output 1–4

Group 2:
Input 5–8 → Output 5–8
```

Now:

$$
4\times4+4\times4
=
32
$$

connections.

---

## General formula

For \(G\) groups:

$$
\boxed{
P=
K^2
\frac{C_{in}C_{out}}{G}
}
$$

versus standard convolution:

$$
P=K^2C_{in}C_{out}
$$

So increasing the number of groups reduces the channel connections and, under comparable dimensions, reduces convolutional parameters/computation.

---

# 6. Group Convolution → Depthwise Convolution

This is a **very important connection**.

Imagine increasing the number of groups:

```text
Standard convolution
G = 1
       ↓
Group convolution
G = 2
       ↓
G = 4
       ↓
G = 8
       ↓
...
       ↓
Depthwise convolution
G = C
```

When:

$$
G=C_{in}
$$

each group contains exactly one input channel.

Therefore:

$$
\boxed{
\text{Depthwise convolution is the extreme grouped-convolution case}
}
$$

This connects directly to your MobileNet/Xception studies.

MobileNet explicitly uses depthwise-separable convolution to construct lightweight CNNs for mobile/embedded vision. ([DOI][3])

---

# 7. Depthwise Convolution

For a \(K\times K\) depthwise convolution:

$$
\boxed{
P=K^2C_{in}
}
$$

Example:

$$
K=3,\qquad C=256
$$

gives:

$$
9\times256
=
\boxed{2,304}
$$

parameters.

Compare standard \(3\times3\):

$$
9\times256\times256
=
589,824
$$

Huge difference.

But remember:

> **Depthwise convolution is cheap because it removes spatial filtering's cross-channel mixing.**

That's why depthwise convolution is normally paired with a **pointwise \(1\times1\) convolution** when channel mixing is needed.

---

# 8. 1×1 Convolution ⭐

A \(1\times1\) convolution is one of the most useful architectural tools.

Suppose:

```text
Input
56 × 56 × 256
```

Apply:

```text
1×1 Conv
256 → 64
```

Output:

```text
56 × 56 × 64
```

Notice:

$$
H,W
$$

stay unchanged.

Only the channel dimension changes.

---

## What is actually happening?

At each spatial location:

```text
256 channel values
       ↓
   1×1 convolution
       ↓
64 channel values
```

So a \(1\times1\) convolution performs **learned channel mixing**.

For:

$$
C_{in}=256,\quad C_{out}=64
$$

parameters:

$$
1\times1\times256\times64
=
\boxed{16,384}
$$

---

## Why is it so useful?

### Channel reduction

$$
256\rightarrow64
$$

### Channel expansion

$$
64\rightarrow256
$$

### Bottleneck

```text
256 → 64 → 256
```

### Shortcut projection

```text
64 channels
    ↓
1×1 Conv
    ↓
128 channels
```

### Feature mixing

It learns combinations across channels.

The Inception family used \(1\times1\) convolutions as part of its strategy for improving computational efficiency while increasing network depth/width. ([arXiv][4])

---

# 9. Large Kernels 🟦

For a long time, \(3\times3\) convolution was an extremely common design choice.

But modern CNN research revisited **large spatial kernels**.

Examples:

$$
3\times3
$$

$$
7\times7
$$

$$
11\times11
$$

etc.

Why?

Because kernel size affects the amount of spatial context available to each operation.

---

# 10. Receptive Field 👁️

The **receptive field** is the region of the input that can influence a particular feature.

Imagine a pixel-level feature:

```text
Input image

┌────────────────────────┐
│                        │
│        object          │
│                        │
└────────────────────────┘
```

A small kernel initially sees:

```text
┌───┐
│   │
└───┘
```

A larger kernel sees:

```text
┌───────────┐
│           │
│           │
└───────────┘
```

So:

$$
K\uparrow
\Rightarrow
\text{local spatial context}\uparrow
$$

---

## But large kernels cost more

For standard convolution:

$$
\text{cost}\propto K^2
$$

Compare:

### 3×3

$$
K^2=9
$$

### 7×7

$$
K^2=49
$$

Therefore:

$$
\frac{49}{9}\approx5.44
$$

So a 7×7 standard convolution has about **5.4× the kernel-area factor** of a 3×3 convolution for the same channel dimensions and output spatial size.

This is why large kernels create a design trade-off.

---

# 11. Modern Large-Kernel Design

An interesting modern example is **ConvNeXt**.

The ConvNeXt paper systematically modernized a ResNet-style ConvNet using ideas inspired partly by contemporary vision architectures. Among the changes was the use of larger \(7\times7\) depthwise convolutions, along with other changes to the block and training recipe. ([CVF Open Access][5])

![Image](https://images.openai.com/static-rsc-4/UXp39L41r_GNNsrH-sVkhrTQArqN6Xq-EuDyMaop3VuPidqLofgI4zmnVvKE0MmHnW6QCKaYiP65EVji7eQtYONhTwpYxUajKM3x3dz1TtQWEurj8tj-j9CZWZe11sWtOS8RfQaGQ9jkv9kuXnTZGaPtKH_iLIzRfGKdi16zfIjoTxnDlAB3vxoVUQKNdcM-?purpose=fullsize)

![Image](https://images.openai.com/static-rsc-4/pfy3olnry_cpz8hYmq84SFOOOX67ySOjQ7i9n6KiykB7PE8xNU1jJZ4ifCQgN9pHJAMWgkLM_uBcOobiFXr9vxgQqPAW1CN3i5DZ5FWWHhNptFA-70jIZ3HNFuRop6_U-OYnymUXms5l2HV5XUzo93oEXkTKU52tb-_GSHcZ5iwpwmIBFuoJJJlKQ0rI4zYJ?purpose=fullsize)

![Image](https://images.openai.com/static-rsc-4/yXGsd9zA_ImZ0LP0RKTVzFBJWX6zfgp0uHeuzfu9ernEexP6NkP7czjshPeefDPVjJZEOBF3vVcNkFRF9fPosW1rqdqXQdk2RoZ4DZJfgmm8I1sWdYIjUD4yG7RFarV5d-QW87GwJy1uEzD2LaBVxpGLE0UOed9_K3AUw2sF2pzmwy2bXwl4qvMg4lxM1YQz?purpose=fullsize)

![Image](https://images.openai.com/static-rsc-4/23s9xWc9nnlpir80vPHxcItH9P1dkQaP5aDJ9dbbLLdKmA-uHH-3PYbJ4aPl3Di1S1g9km_zuLu-gQLkWVmTmHHVfbDQa-SPoN6bZOwutffW3j4bKY2thhuZDLotzceyP1F5Kx14AxccVzjbahMROHu3tCO4iUOLhC8DAkZydZjUEbgkgfLVuAjJrePMGQz7?purpose=fullsize)

![Image](https://images.openai.com/static-rsc-4/4B5ieVV_p3FVYNhXKhLSLqQ2gjtMl5y-J3mP8zrp6HCShfPZurUzMSqHfssciWcL7vnvILc9PfsRTOogNPApST90WpHwxPHfYIiJZXbJzzizBIpu3eOY5jWDT-AuBL7FlJIVT2oCZk4k_3efBrhVd30CJQRa5bd0PYPHLWOKJEcF5UCkoTaGUAU3yS4otHjq?purpose=fullsize)

The important lesson is:

> **Large kernels don't automatically mean inefficient CNNs. Their cost depends heavily on how the convolution is implemented.**

A large **depthwise** convolution is much cheaper than a large standard convolution because it doesn't perform full input-output channel mixing.

---

# 12. Global Average Pooling 🌐

Suppose the CNN produces:

$$
7\times7\times512
$$

feature maps.

A traditional approach could flatten them:

$$
7\times7\times512
=
25,088
$$

values.

Instead, **Global Average Pooling (GAP)** averages each channel's entire spatial map.

For channel \(c\):

$$
\boxed{
z_c=
\frac{1}{HW}
\sum_{i=1}^{H}
\sum_{j=1}^{W}
x_{c,i,j}
}
$$

Therefore:

$$
H\times W\times C
\rightarrow
1\times1\times C
$$

Example:

```text
7 × 7 × 512
      ↓
     GAP
      ↓
1 × 1 × 512
      ↓
     512
```

![Image](https://images.openai.com/static-rsc-4/xsG8B_1mjzTz9cK9zSwWqKuJVlVTnQV9z7xMwre0UjDzJeGo0eveDeATKiidAPmJ6P5KLPEYR8H4cewsGLBgR6LuyorwzlAfEV94AL2waF1oMO-sXVOi1hwPDok-k2NqZS_s9qBnXw7OTe4BLApx6aalbW_QbCB0lRXCqX2PS6vQn_wgFGE8PskpT5F2bNbt?purpose=fullsize)

![Image](https://images.openai.com/static-rsc-4/3OaK8-oD8x8yf6rn6Y-0Mf6KVgOMlIVaPqRu1FFH2iBWDI1g-PtZoB7GuMizLlcs7nZOVQGi5X_DKE9dCuJX_xpKst3RY9Jzobi1OsOyENF-UioPYX73Vl1ZMlnMKjW84zfc2ym3RU_X91Qjfp1ZSAhHVJSCyZem0MYeIpT53d0oItmxHlGGXq67_rP5c_y-?purpose=fullsize)

![Image](https://images.openai.com/static-rsc-4/VJEYYERu2fzl20BuIU7cQq3gP0yGntTEIvHXFBMK_wSVYnqeqzUqA-mSNOCeNJtiUDmsul2POZkXtxd2V1RSmjGEicL4oxkTXI__QEYospTZxHCmZX7jtSgvrSRHIPeQpTV-TeknAxzGWjaSioFF6l-P-74zbagMAQsUSiaL4u7LuuwMKbe_So3QVLWBWkkH?purpose=fullsize)

![Image](https://images.openai.com/static-rsc-4/7Ie77snMNLqDxf_6K9KiIN4lzN8QrLgWTOYc4T-xceUJjDjJ0Wfv0wihOdlZSmn-SJ01sxRao4Rd8PESfJatfkyGlRhw_DB9Zl9Q_2-OhFG0tBJbxlLCqmMxBlZgxPgeMJSR9tkIqPxr3dsEz9EmSuX-8CfoS5kWMphb_XF0ptN5EohXcwM7KSPXyUy9LCrI?purpose=fullsize)

![Image](https://images.openai.com/static-rsc-4/XnvSXAHjs4FP5eZr2xdkFU5zpxXoFKCr2k_8OWRLn97Oo8ogEp0HLKSU_yqkwzNamteW_52Z1ZgyBW-nKGeNanHigRCjz9WI9zEYOhNtmSnh9ZAkebkSDLNZjRaJZtI3Kno0mhMNSFbJLQz3GzAGZMzABfTd_DW59bPGvgDPGUIGE6E0sclfrpNx5sxiumGT?purpose=fullsize)

![Image](https://images.openai.com/static-rsc-4/dsFJClMo78yRw0CuEK0BNPzXzTeGXgLKM2ZgJancWVifA2dpBlaGt4R5PfmnpO805ynGuODhlPFpcvIlaWPsH08Wmyc7v_4L01aWXwqK-7p4wSmSYC32Ti65ZPb_LWtxssDdHF6_IxEHS7aoRUCf6OdWTCibVnjFNroRD31Lo2LquvwqpgghXefJz3inFfex?purpose=fullsize)

---

## Numerical example

Suppose one feature map is:

$$
\begin{bmatrix}
2&4\\
6&8
\end{bmatrix}
$$

GAP:

$$
\frac{2+4+6+8}{4}
=
\boxed{5}
$$

So:

```text
2  4
6  8
 ↓
 5
```

Each channel produces **one number**.

---

# 13. Why GAP Is Useful

Suppose we have:

$$
7\times7\times512
$$

Flattening gives:

$$
25,088
$$

features.

A fully connected layer with 1000 outputs would require:

$$
25,088\times1000
$$

$$
=\boxed{25.088\text{ million}}
$$

weights, ignoring bias.

GAP instead gives:

$$
512
$$

values before classification.

A 512 → 1000 classifier needs:

$$
512,000
$$

weights.

🔥 That's a massive reduction.

---

# 14. Design Dimensions

Now move from **building blocks** to the dimensions we can manipulate.

```text
Depth
Width
Resolution
Receptive field
Parameters
FLOPs
Memory
Latency
```

These aren't independent.

Changing one often affects several others.

---

# 15. Depth

Depth asks:

> **How many transformations does the network perform?**

```text
Input
 ↓
Block
 ↓
Block
 ↓
Block
 ↓
Block
 ↓
Output
```

More depth can provide more hierarchical transformations.

But:

$$
\text{Depth}\uparrow
\Rightarrow
\text{computation}\uparrow
$$

and potentially:

$$
\text{latency}\uparrow
$$

Residual connections helped make very deep CNNs easier to optimize; the original ResNet work demonstrated networks reaching 152 layers in its ImageNet experiments. ([CVF Open Access][1])

---

# 16. Width

Width is primarily the number of channels.

```text
56 × 56 × 64
          ↑
        width
```

versus:

```text
56 × 56 × 128
           ↑
         width
```

Increasing width generally gives more feature channels/capacity, but also tends to increase:

* Parameters
* FLOPs
* Activation memory

---

# 17. Resolution

Resolution refers to spatial dimensions:

$$
H\times W
$$

Example:

```text
224 × 224
```

versus:

```text
448 × 448
```

If both dimensions double:

$$
\frac{448^2}{224^2}=4
$$

So you now have **4× as many spatial positions**.

This is why resolution is expensive.

A useful mental model:

```text
Resolution ×2 in H and W
          ↓
4× spatial positions
          ↓
Potentially much more computation/memory
```

This is also why EfficientNet treated resolution as one of its three explicit scaling dimensions.

---

# 18. Receptive Field

Receptive field answers:

> **How much of the original input can affect a particular feature?**

It can be increased through:

* More layers
* Larger kernels
* Downsampling
* Dilated convolutions

So:

```text
Depth
  +
Kernel size
  +
Downsampling
  +
Dilation
      ↓
Receptive field
```

But these methods have different computational consequences.

---

# 19. Parameters

For standard convolution:

$$
\boxed{
P=K^2C_{in}C_{out}
}
$$

Parameters determine the number of learned weights.

If weights use 32-bit floating point:

$$
\text{Memory}
\approx
P\times4\text{ bytes}
$$

For example:

$$
25M\times4
=
100MB
$$

just for the weights.

⚠️ Actual training memory is much larger because of activations, gradients, optimizer states, etc.

---

# 20. FLOPs

FLOPs are a measure of theoretical computational work.

For a standard convolution, ignoring exact convention details:

$$
\text{MACs}
=
HWC_{in}C_{out}K^2
$$

and some conventions count one multiply-add as:

$$
2\text{ FLOPs}
$$

Therefore always check whether a source is reporting **MACs or FLOPs**.

This matters when comparing papers.

---

# 21. Memory 🧠

Memory isn't just the model weights.

There are at least two important categories:

### Parameter memory

```text
Weights
```

### Activation memory

```text
Input
 ↓
Activation
 ↓
Activation
 ↓
Activation
 ↓
Output
```

During training, many intermediate activations are retained because they are needed for backpropagation.

Therefore:

$$
\boxed{
\text{Small parameter count}
\neq
\text{small total memory usage}
}
$$

---

# 22. Latency ⚡

Latency means:

> **How long does an operation/model actually take to execute?**

This is where architecture analysis becomes practical.

Latency depends on:

```text
FLOPs
+
Memory access
+
Memory bandwidth
+
Hardware
+
Parallelism
+
Kernel implementation
+
Software stack
+
Batch size
```

Therefore:

$$
\boxed{
\text{FLOPs}\neq\text{Latency}
}
$$

This is a lesson you've already seen with MobileNet.

Research on mobile depthwise/pointwise convolution implementations has shown that implementation details and hardware utilization can substantially affect actual performance. ([arXiv][6])

---

# 🔥 23. The Accuracy–Efficiency Trade-off

Your syllabus represents this as:

```text
Accuracy
   ↕
Parameters
   ↕
FLOPs
   ↕
Memory
   ↕
Latency
```

But technically, it's better to think of this as a **multi-objective design problem** rather than a literal chain.

For example:

```text
                 Accuracy
                    ↑
                    │
                    │
                    │       ●
                    │
                    │  ●
                    │
                    │ ●
                    └────────────────→ Compute
```

You may have one model with:

```text
Higher accuracy
Higher FLOPs
Higher latency
```

and another with:

```text
Lower accuracy
Lower FLOPs
Lower latency
```

Neither description alone tells you which one is appropriate.

The **deployment requirement** determines what matters.

---

# 24. Same Model, Different Application

### Data-center inference

```text
Large GPU
     ↓
High memory
     ↓
High throughput
     ↓
Larger model may be acceptable
```

### Mobile inference

```text
Limited compute
     ↓
Limited memory
     ↓
Battery constraints
     ↓
Latency matters heavily
```

### Edge device

```text
Limited hardware
     ↓
Memory + latency + power
     ↓
Architecture must be hardware-aware
```

So architecture design is always contextual.

---

# 🧩 25. Putting the Building Blocks Together

Consider a hypothetical modern CNN block:

```text
Input
  │
  ▼
1×1 Conv
  │
  ▼
Bottleneck
  │
  ▼
Large-kernel Depthwise Conv
  │
  ▼
1×1 Conv
  │
  ├─────────────────┐
  │                 │
  │             Shortcut
  │                 │
  └────────── + ◄───┘
             │
             ▼
           Output
```

Now we can explain **why every component exists**:

| Component    | Design problem                                        |
| ------------ | ----------------------------------------------------- |
| \(1\times1\) | Channel mixing / dimension adjustment                 |
| Bottleneck   | Reduce expensive computation                          |
| Depthwise    | Cheap spatial filtering                               |
| Large kernel | Increase spatial context                              |
| Skip         | Information/gradient pathway                          |
| GAP          | Compress spatial representation before classification |

🔥 **This is the skill Module 24 is teaching.**

---

# 26. The Historical Connection

You can now see how individual ideas evolved:

```text
                 CNN DESIGN IDEAS
                       │
        ┌──────────────┼───────────────┐
        ↓              ↓               ↓
   Efficient         Information     Context
   computation       flow/reuse      expansion
        │              │               │
   ┌────┼────┐      ┌──┼──┐         ┌──┴──┐
   ↓    ↓    ↓      ↓  ↓  ↓         ↓     ↓
 1×1  Group Depth  Skip Dense     Large  Deep
      Conv  wise   connection      kernel network
```

And famous architectures can be viewed as combinations of these principles:

```text
Inception
   ↓
Multi-branch + 1×1 + efficient computation

ResNet
   ↓
Residual / skip connections

DenseNet
   ↓
Dense feature reuse

ResNeXt
   ↓
Grouped transformations / cardinality

MobileNet
   ↓
Depthwise-separable convolution

Xception
   ↓
Extreme spatial/channel separation

EfficientNet
   ↓
Depth + width + resolution scaling

RegNet
   ↓
Systematic architecture design

ConvNeXt
   ↓
Modernized pure ConvNet design
```

In particular, ResNeXt explicitly introduced **cardinality**—the number of parallel transformations—as an additional design dimension alongside depth and width. ([CVF Open Access][7])

---

# 🔥 27. One Very Important Modern Insight

Architecture design isn't just:

> **Make the network bigger.**

Modern CNN research repeatedly asks:

> **Where should the computational budget be spent?**

For example:

```text
Available compute
      ↓
Should we increase:
      │
      ├── Depth?
      │
      ├── Width?
      │
      ├── Resolution?
      │
      ├── Receptive field?
      │
      ├── Number of branches?
      │
      └── Feature reuse?
```

This connects directly to:

* EfficientNet → compound scaling
* RegNet → structured design spaces
* ResNeXt → cardinality
* ConvNeXt → systematic modernization of ConvNet design

---

# 🧠 28. How to Analyze Any New CNN

This is probably the **most valuable checklist from Module 24**.

When you encounter an unfamiliar CNN, ask:

### Architecture

**1. What is the depth?**

$$
\text{How many transformations?}
$$

**2. What is the width?**

$$
\text{How many channels?}
$$

**3. What are the stages?**

$$
\text{Where does resolution change?}
$$

---

### Computation

**4. What convolution is being used?**

```text
Standard?
Group?
Depthwise?
```

**5. Are there bottlenecks?**

```text
Compress → process → expand?
```

**6. Where are 1×1 convolutions used?**

```text
Reduction?
Expansion?
Projection?
Mixing?
```

---

### Information flow

**7. Are there skip connections?**

$$
F(X)+X
$$

**8. Are there dense connections?**

$$
[X_0,X_1,\ldots]
$$

---

### Spatial context

**9. What is the receptive field?**

Look at:

```text
Kernel size
Depth
Downsampling
Dilation
```

---

### Efficiency

**10. How many parameters?**

**11. How many FLOPs/MACs?**

**12. How much activation memory?**

**13. What is actual latency on target hardware?**

---

### Final question

> **What trade-off was the designer trying to make?**

🔥 If you can answer that, you are no longer just memorizing CNN architectures.

You're **analyzing them**.

---

# 📌 29. High-Value Formula Sheet

### Standard convolution

$$
\boxed{
P=K^2C_{in}C_{out}
}
$$

---

### Group convolution

$$
\boxed{
P=
K^2\frac{C_{in}C_{out}}{G}
}
$$

---

### Depthwise convolution

$$
\boxed{
P=K^2C_{in}
}
$$

---

### 1×1 convolution

$$
\boxed{
P=C_{in}C_{out}
}
$$

---

### GAP

$$
\boxed{
z_c=
\frac{1}{HW}
\sum_i\sum_j x_{cij}
}
$$

---

### Residual connection

$$
\boxed{
Y=F(X)+X
}
$$

---

### Dense connection

$$
\boxed{
X_l=H_l([X_0,X_1,\ldots,X_{l-1}])
}
$$

---

### Standard convolution MACs

$$
\boxed{
MACs\approx HWC_{in}C_{out}K^2
}
$$

---

# 🧠 30. The Ultimate Cheat Sheet

```text
╔══════════════════════════════════════════════╗
║      MODERN CNN DESIGN PRINCIPLES            ║
╠══════════════════════════════════════════════╣
║                                              ║
║ 1×1 CONV                                     ║
║ → channel mixing / dimension change          ║
║                                              ║
║ BOTTLENECK                                   ║
║ → compress → expensive operation → expand    ║
║                                              ║
║ SKIP CONNECTION                              ║
║ → F(X) + X                                   ║
║ → easier information/gradient path            ║
║                                              ║
║ DENSE CONNECTION                             ║
║ → concatenate previous features               ║
║ → feature reuse                               ║
║                                              ║
║ GROUP CONV                                   ║
║ → split channels into groups                  ║
║                                              ║
║ DEPTHWISE CONV                               ║
║ → one spatial filter/channel                  ║
║ → extreme grouped convolution                ║
║                                              ║
║ LARGE KERNEL                                 ║
║ → larger spatial context                      ║
║ → potentially more computation                ║
║                                              ║
║ GAP                                          ║
║ → H×W×C → 1×1×C                             ║
║ → compact global representation               ║
║                                              ╠══════════════════════════════════════════════╣
║ DESIGN DIMENSIONS                            ║
║                                              ║
║ Depth       → transformations                 ║
║ Width       → channels                        ║
║ Resolution  → spatial detail                  ║
║ Receptive   → input context                   ║
║ Parameters  → learned weights                 ║
║ FLOPs       → theoretical computation         ║
║ Memory      → weights + activations           ║
║ Latency     → actual execution time           ║
║                                              ╠══════════════════════════════════════════════╣
║ CORE QUESTION                                ║
║                                              ║
║ "WHY was this architectural choice made?"   ║
║                                              ╚══════════════════════════════════════════════╝
```

---

# 🔥 Final Mental Model

Don't memorize this module as eight independent definitions.

Think of CNN design as a **toolbox**:

```text
                    CNN DESIGN TOOLBOX
                           │
       ┌───────────────────┼───────────────────┐
       ↓                   ↓                   ↓
   SAVE COMPUTE       IMPROVE FLOW        EXPAND CONTEXT
       │                   │                   │
  1×1 Conv             Skip                 Large kernel
  Group Conv           Dense                Depth
  Depthwise
  Bottleneck
       │                   │                   │
       └───────────────────┼───────────────────┘
                           ↓
                  BALANCE DESIGN
                           │
          ┌────────────────┼────────────────┐
          ↓                ↓                ↓
       Accuracy          Memory           Latency
          ↓                ↓                ↓
                   Target Hardware
```

And the **one sentence I want you to remember** from Module 24 is:

> 🧠 **Modern CNN architecture design is the art of allocating limited computational and memory resources across depth, width, resolution, receptive field, connectivity, and efficient operations to achieve the required accuracy and deployment performance.**

That is the conceptual foundation behind a *lot* of modern CNN architecture research. 🔥

---

## 🔗 Research & Visual References

* [ResNet — Deep Residual Learning for Image Recognition](https://openaccess.thecvf.com/content_cvpr_2016/html/He_Deep_Residual_Learning_CVPR_2016_paper.html?utm_source=chatgpt.com) — original residual-learning paper. ([CVF Open Access][1])
* [DenseNet — Densely Connected Convolutional Networks](https://openaccess.thecvf.com/content_cvpr_2017/html/Huang_Densely_Connected_Convolutional_CVPR_2017_paper.html?utm_source=chatgpt.com) — original dense-connectivity paper. ([CVF Open Access][8])
* [ResNeXt — Aggregated Residual Transformations](https://openaccess.thecvf.com/content_cvpr_2017/html/Xie_Aggregated_Residual_Transformations_CVPR_2017_paper.html?utm_source=chatgpt.com) — grouped transformations/cardinality. ([CVF Open Access][7])
* [MobileNet — Efficient Convolutional Neural Networks for Mobile Vision Applications](https://arxiv.org/abs/1704.04861?utm_source=chatgpt.com) — depthwise-separable convolution and efficiency. ([DOI][3])
* [Inception — Going Deeper with Convolutions](https://arxiv.org/abs/1409.4842?utm_source=chatgpt.com) — multi-branch design and efficient computation. ([arXiv][4])
* [ConvNeXt — A ConvNet for the 2020s](https://openaccess.thecvf.com/content/CVPR2022/html/Liu_A_ConvNet_for_the_2020s_CVPR_2022_paper.html?utm_source=chatgpt.com) — modern ConvNet design and large-kernel depthwise convolution. ([CVF Open Access][5])

---